# Отчёт 3. Оптимизаторы машинного обучения и линейный SVM

Сравнение SGD, Momentum, Nesterov, AdaGrad, RMSProp и Adam, исследование размера батча и регуляризации, а также реализация линейного SVM через SGD.

> **Воспроизводимость.** Для воспроизведения перезапустите ядро и выполните все ячейки по порядку.

[Описание, результаты и инструкция по запуску](../docs/03-ml-optimizers.md)


# Базовая часть и Extension-1


## Код для запуска

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import tracemalloc
import os
import torch

os.makedirs('results', exist_ok=True)


class BaseOptimizer:
    def __init__(self, lr=0.01):
        self.lr = lr
        self.name = "Base"

    def step(self, w, grad):
        raise NotImplementedError

    def reset(self):
        pass


class CustomSGD(BaseOptimizer):
    def __init__(self, lr=0.01):
        super().__init__(lr)
        self.name = "SGD"

    def step(self, w, grad):
        return w - self.lr * grad


class CustomMomentum(BaseOptimizer):
    def __init__(self, lr=0.01, momentum=0.9):
        super().__init__(lr)
        self.momentum = momentum
        self.velocity = None
        self.name = "Momentum"

    def step(self, w, grad):
        if self.velocity is None:
            self.velocity = np.zeros_like(w)
        self.velocity = self.momentum * self.velocity - self.lr * grad
        return w + self.velocity

    def reset(self):
        self.velocity = None


class TorchWrapper(BaseOptimizer):
    def __init__(self, opt_class, lr=0.01, **kwargs):
        super().__init__(lr)
        self.opt_class = opt_class
        self.kwargs = kwargs
        self.param = None
        self.opt = None
        self.name = opt_class.__name__

    def step(self, w, grad):
        if self.opt is None:
            self.param = torch.tensor(w, dtype=torch.float32, requires_grad=True)
            self.opt = self.opt_class([self.param], lr=self.lr, **self.kwargs)
        else:
            with torch.no_grad():
                self.param.copy_(torch.tensor(w, dtype=torch.float32))
        self.opt.zero_grad()
        self.param.grad = torch.tensor(grad, dtype=torch.float32)
        self.opt.step()
        return self.param.detach().numpy()

    def reset(self):
        self.param = None
        self.opt = None


def generate_quadratic_problem(n_features=2, n_samples=500, seed=42):
    np.random.seed(seed)
    X = np.random.randn(n_samples, n_features)
    true_w = np.array([1.5, -2.0])[:n_features]
    y = np.dot(X, true_w) + np.random.randn(n_samples) * 0.1
    return X, y, true_w


def compute_loss_and_grad(X_batch, y_batch, w, l1_reg=0.0, l2_reg=0.0, elastic_alpha=0.0):
    diff = np.dot(X_batch, w) - y_batch
    mse = np.mean(diff ** 2)
    grad = (2.0 / len(y_batch)) * np.dot(X_batch.T, diff)
    reg_loss = 0.0
    if l1_reg > 0:
        reg_loss += l1_reg * np.sum(np.abs(w))
        grad += l1_reg * np.sign(w)
    if l2_reg > 0:
        reg_loss += l2_reg * np.sum(w ** 2)
        grad += 2 * l2_reg * w
    if elastic_alpha > 0 and l2_reg > 0:
        reg_loss += elastic_alpha * l2_reg * np.sum(np.abs(w))
        grad += elastic_alpha * l2_reg * np.sign(w)
    return mse + reg_loss, grad


def optimize(objective_func, X, y, optimizer, w0, epochs=100, batch_size=32, l1_reg=0.0, l2_reg=0.0, elastic_alpha=0.0):
    optimizer.reset()
    w = w0.copy()
    n_samples = len(y)
    losses = []
    trajectory = [w.copy()]
    tracemalloc.start()
    start_time = time.time()
    flops = 0
    for epoch in range(epochs):
        indices = np.random.choice(n_samples, size=min(batch_size, n_samples), replace=False)
        X_batch, y_batch = X[indices], y[indices]
        loss, grad = objective_func(X_batch, y_batch, w, l1_reg, l2_reg, elastic_alpha)
        w = optimizer.step(w, grad)
        losses.append(loss)
        trajectory.append(w.copy())
        flops += 2 * len(y_batch) * len(w) + len(w)
    end_time = time.time()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    final_loss, _ = objective_func(X, y, w, l1_reg, l2_reg, elastic_alpha)
    return {
        'w_final': w,
        'final_loss': final_loss,
        'losses': losses,
        'trajectory': np.array(trajectory),
        'name': optimizer.name,
        'time_sec': end_time - start_time,
        'peak_memory_mb': peak_mem / (1024 * 1024),
        'flops': flops
    }


def plot_convergence(results):
    plt.figure(figsize=(12, 6))
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
    for idx, res in enumerate(results):
        plt.semilogy(res['losses'], label=res['name'], color=colors[idx % len(colors)], linewidth=2)
    plt.xlabel('Итерация')
    plt.ylabel('Значение функции (log)')
    plt.title('Сходимость методов')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('results/convergence.png', dpi=150)
    plt.close()


def plot_2d_trajectory(results, X, y, true_w, l1=0.0, l2=0.0, elastic=0.0, filename="trajectories.png",
                       title="Траектории"):
    w1_range = np.linspace(true_w[0] - 2.0, true_w[0] + 2.0, 100)
    w2_range = np.linspace(true_w[1] - 2.0, true_w[1] + 2.0, 100)
    W1, W2 = np.meshgrid(w1_range, w2_range)
    Z = np.zeros_like(W1)
    for i in range(W1.shape[0]):
        for j in range(W1.shape[1]):
            w_test = np.array([W1[i, j], W2[i, j]])
            Z[i, j], _ = compute_loss_and_grad(X, y, w_test, l1_reg=l1, l2_reg=l2, elastic_alpha=elastic)
    plt.figure(figsize=(12, 10))
    colors = ['blue', 'red', 'green', 'orange', 'purple', 'brown']
    contour = plt.contour(W1, W2, Z, levels=40, cmap='viridis', alpha=0.4)
    plt.colorbar(contour, label='Значение функции')
    for idx, res in enumerate(results):
        traj = res['trajectory']
        plt.plot(traj[:, 0], traj[:, 1], 'o-', markersize=3, markerfacecolor='black',
                 markeredgecolor='black', color=colors[idx % len(colors)], linewidth=1.5, alpha=0.8, label=res['name'])
    plt.plot(true_w[0], true_w[1], 'ks', markersize=6, label='Минимум', zorder=10)
    plt.xlabel('Параметр w_1')
    plt.ylabel('Параметр w_2')
    plt.title(title)
    plt.legend(fontsize=10, loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'results/{filename}', dpi=150)
    plt.close()


def print_table(results, title=""):
    if title:
        print(title)
    print(f"{'Метод':<15} {'Потеря':<12} {'Время, с':<10} {'Память, МБ':<12}")
    for res in results:
        print(f"{res['name']:<15} {res['final_loss']:<12.6f} {res['time_sec']:<10.4f} {res['peak_memory_mb']:<12.4f}")
    print()


def main():
    X, y, true_w = generate_quadratic_problem(n_features=2, n_samples=500)
    w0 = np.array([0.0, 0.0])

    optimizers_to_test = [
        CustomSGD(lr=0.05),
        CustomMomentum(lr=0.05, momentum=0.9),
        TorchWrapper(torch.optim.SGD, lr=0.05, momentum=0.9, nesterov=True),
        TorchWrapper(torch.optim.Adagrad, lr=0.2),
        TorchWrapper(torch.optim.RMSprop, lr=0.05),
        TorchWrapper(torch.optim.Adam, lr=0.05)
    ]

    results = []
    for opt in optimizers_to_test:
        res = optimize(compute_loss_and_grad, X, y, opt, w0, epochs=100, batch_size=32)
        results.append(res)

    print_table(results, title="Сравнение методов (batch=32, 100 эпох)")
    plot_convergence(results)
    plot_2d_trajectory(results, X, y, true_w, filename="trajectories_base.png", title="Траектории на линиях уровня")

    batch_sizes = [1, 8, 32, 128, 500]
    batch_results = []
    for bs in batch_sizes:
        res_sgd = optimize(compute_loss_and_grad, X, y, CustomSGD(lr=0.05), w0, epochs=100, batch_size=bs)
        res_sgd['name'] = f"SGD (bs={bs})"
        batch_results.append(res_sgd)
        res_mom = optimize(compute_loss_and_grad, X, y, CustomMomentum(lr=0.05, momentum=0.9), w0, epochs=100,
                           batch_size=bs)
        res_mom['name'] = f"Momentum (bs={bs})"
        batch_results.append(res_mom)

    print_table(batch_results, title="Влияние размера батча")

    reg_configs = [
        ("Без рег.", 0.0, 0.0, 0.0),
        ("L1", 0.1, 0.0, 0.0),
        ("L2", 0.0, 0.1, 0.0),
        ("ElasticNet", 0.05, 0.05, 0.5)
    ]

    reg_results = []
    for name, l1, l2, elastic in reg_configs:
        opt = CustomMomentum(lr=0.05, momentum=0.9)
        res = optimize(compute_loss_and_grad, X, y, opt, w0, epochs=100, batch_size=32, l1_reg=l1, l2_reg=l2,
                       elastic_alpha=elastic)
        res['name'] = name
        reg_results.append(res)

    print_table(reg_results, title="Влияние регуляризации")

    opt_elastic = CustomMomentum(lr=0.05, momentum=0.9)
    res_elastic = optimize(compute_loss_and_grad, X, y, opt_elastic, w0, epochs=100, batch_size=32, l1_reg=0.05,
                           l2_reg=0.05, elastic_alpha=0.5)
    plot_2d_trajectory([res_elastic], X, y, true_w, l1=0.05, l2=0.05, elastic=0.5, filename="trajectories_reg.png",
                       title="Траектория с Elastic Net")


if __name__ == "__main__":
    main()


| Метод | Потеря | Время, с | Память, МБ |
| --- | --- | --- | --- |
| SGD | 0.010291 | 0.0159 | 0.0241 |
| Momentum | 0.010477 | 0.0166 | 0.0233 |
| Nesterov | 0.010861 | 0.0187 | 0.0233 |
| AdaGrad | 0.011642 | 0.0180 | 0.0240 |
| RMSProp | 0.010340 | 0.0188 | 0.0233 |
| Adam | 0.010885 | 0.0224 | 0.0240 |

| Метод | Потеря | Время, с | Память, МБ |
| --- | --- | --- | --- |
| SGD (bs=1) | 0.010285 | 0.0149 | 0.0218 |
| Momentum (bs=1) | 0.499017 | 0.0159 | 0.0219 |
| SGD (bs=8) | 0.010529 | 0.0180 | 0.0221 |
| Momentum (bs=8) | 0.011788 | 0.0187 | 0.0222 |
| SGD (bs=32) | 0.010296 | 0.0164 | 0.0232 |
| Momentum (bs=32) | 0.010553 | 0.0158 | 0.0233 |
| SGD (bs=128) | 0.010269 | 0.0157 | 0.0284 |
| Momentum (bs=128) | 0.010287 | 0.0166 | 0.0285 |
| SGD (bs=500) | 0.010262 | 0.0178 | 0.0482 |
| Momentum (bs=500) | 0.010366 | 0.0190 | 0.0483 |

| Метод | Потеря | Время, с | Память, МБ |
| --- | --- | --- | --- |
| Без рег. | 0.010320 | 0.0165 | 0.0233 |
| L1 | 0.354685 | 0.0195 | 0.0233 |
| L2 | 0.580335 | 0.0195 | 0.0233 |
| ElasticNet | 0.554271 | 0.0258 | 0.0233 |

# Вывод 


Вывод:
Все методы показывают близкие результаты по итоговой ошибке. Обычный SGD и RMSProp сходятся лучше остальных. Адаптивные методы работают чуть медленнее, но дают хороший результат.
Размер батча сильно влияет на стабильность. Обычный SGD работает хорошо при любом размере, а Momentum при batch_size=1 расходится из-за накопления шума градиента. С ростом размера батча оба метода работают стабильнее, лучшие результаты достигаются при batch_size=500.
Регуляризация с коэффициентом 0.1 сильно ухудшает результат, увеличивая ошибку в десятки раз. Для данной задачи нужны гораздо меньшие значения коэффициентов.

# Extension-2


## Код для запуска

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs('results', exist_ok=True)

np.random.seed(42)
n = 200
X_pos = np.random.randn(n // 2, 2) + np.array([2, 2])
X_neg = np.random.randn(n // 2, 2) + np.array([-2, -2])
X = np.vstack([X_pos, X_neg])
y = np.hstack([np.ones(n // 2), -np.ones(n // 2)])

w = np.zeros(2)
b = 0.0
lr = 0.01
C = 1.0
epochs = 500
losses = []

for epoch in range(epochs):
    idx = np.random.randint(0, n)
    xi, yi = X[idx], y[idx]
    margin = yi * (np.dot(w, xi) + b)
    if margin < 1:
        grad_w = w - C * yi * xi
        grad_b = -C * yi
    else:
        grad_w = w
        grad_b = 0.0
    w = w - lr * grad_w
    b = b - lr * grad_b
    hinge = max(0, 1 - margin)
    loss = 0.5 * np.dot(w, w) + C * hinge
    losses.append(loss)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(losses)
plt.xlabel('Итерация')
plt.ylabel('Значение функции')
plt.title('Сходимость SVM через SGD')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.scatter(X_pos[:, 0], X_pos[:, 1], c='blue', label='+1', alpha=0.6)
plt.scatter(X_neg[:, 0], X_neg[:, 1], c='red', label='-1', alpha=0.6)
x_line = np.linspace(-5, 5, 100)
y_line = -(w[0] * x_line + b) / w[1]
y_margin_pos = -(w[0] * x_line + b - 1) / w[1]
y_margin_neg = -(w[0] * x_line + b + 1) / w[1]
plt.plot(x_line, y_line, 'k-', label='Разделяющая линия')
plt.plot(x_line, y_margin_pos, 'k--', alpha=0.5)
plt.plot(x_line, y_margin_neg, 'k--', alpha=0.5)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('Разделяющая гиперплоскость SVM')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/svm.png', dpi=150)
plt.close()

print(f"w = {w}")
print(f"b = {b:.4f}")
print(f"Финальное значение функции: {losses[-1]:.4f}")

## Комментраий
Линейный SVM ищет разделяющую гиперплоскость w·x + b = 0, которая максимизирует зазор между классами. Это сводится к задаче:

minimize[ (1/2)||w||² + C · Σ max(0, 1 − yᵢ(w·xᵢ + b)) ]

Первое слагаемое — L2-регуляризация, которая максимизирует зазор. Второе — hinge loss, штрафующая за ошибки и объекты внутри зазора. Параметр C управляет балансом между шириной зазора и количеством нарушений.

Градиент для одного объекта (xᵢ, yᵢ):
- Если yᵢ(w·xᵢ + b) < 1: ∇w = w − C·yᵢxᵢ, ∇b = −C·yᵢ
- Иначе: ∇w = w, ∇b = 0

На каждом шаге берётся один случайный объект, считается градиент, параметры обновляются.